In [2]:
from dotenv import load_dotenv
load_dotenv()

True

---

#### 문서 로드

In [3]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/KCI_FI003153549_p5.pdf")
documents = loader.load()

#### 문서 분할

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

#### 임베딩 모델(캐싱)

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace = underlying_embeddings.model
)

#### 임베딩 & FAISS(Facebook AI Similarity Search) 벡터스토어 생성 및 저장

##### Case1. In-memory

In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

##### Case2. 로컬 디스크 저장(기존 파일 삭제 후 저장)

In [33]:
vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

# 영구적인 파일(persistent file)**로 디스크에 저장
# 기존 폴더에 새로운 인덱스 파일을 덮어쓰기 때문에 중복된 파일이 생성되지 않음(항상 가장 마지막에 저장된 벡터스토어의 파일만 존재)
vectorstore.save_local("./faiss_index")

In [34]:
vectorstore

In [36]:
vectorstore = None

In [37]:
vectorstore

In [ ]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    cached_embedder,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [40]:
vectorstore

##### Case3. 로컬 디스크 저장(기존 파일이 있을 경우 로드)

In [15]:
vectorstore = None

In [16]:
vectorstore

In [ ]:
FAISS_INDEX_PATH = "./faiss_index"

if os.path.exists(FAISS_INDEX_PATH):
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embedding_model,
        allow_dangerous_deserialization=True,
    )
else:
    # FAISS 벡터스토어 생성 및 저장
    vectorstore = FAISS.from_documents(splitted_documents, embedding_model)
    vectorstore.save_local(FAISS_INDEX_PATH)

In [ ]:
vectorstore

---

In [22]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

In [ ]:
results = vectorstore.similarity_search(query, k=5) # 검색을 외부에서 미리 실행한 후 반환된 결과 사용

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    '''다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트:{context}

질문: {question}
'''
)

prompt

In [ ]:
# 질문 예시
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
)

chain = prompt | llm | StrOutputParser()

In [ ]:
response = chain.invoke({'context': results, 'question': query})

In [ ]:
print(response)

---

In [ ]:
# 리트리버 생성
retriever = vectorstore.as_retriever()

In [ ]:
retriever.invoke(query)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_api_key)

# retriever, RunnablePassthrough 객체 전달
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

In [ ]:
response = chain.invoke(query) # query는 RunnablePassthrough()를 통과하여 question이라는 키의 값이 됨

In [ ]:
print(response)